In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent))

import numpy as np
from pyqpanda3.hamiltonian import PauliOperator
from pyqpanda3.core import QProg
from pyqpanda_alg.VQE import VQE
from pyqpanda_alg.VQE.vqe import hardware_efficient_circuit

# VQE Demo 2: Hardware-Efficient Ansatz

This notebook demonstrates the hardware-efficient ansatz and solves
a two-qubit Hamiltonian:

$$H = -Z_0 - Z_1 + 0.5 \cdot Z_0 Z_1$$

Ground state $|00\rangle$:

$$E = \langle 00|H|00\rangle = (-1) + (-1) + 0.5 = -1.5$$

In [ ]:
# Visualize the hardware-efficient ansatz circuit (depth=2, 3 qubits)
n_qubits = 3
q = QProg(n_qubits).qubits()
params = [0.1] * (2 * n_qubits * 2)
cir = hardware_efficient_circuit(q, params, depth=2)
print(cir)

In [ ]:
# Two-qubit Hamiltonian: negative Z terms so |00⟩ is the ground state
H = (PauliOperator({"Z0": -1.0})
     + PauliOperator({"Z1": -1.0})
     + PauliOperator({"Z0 Z1": 0.5}))
print(f"Hamiltonian: {H}")
print(f"Exact ground-state energy: -1.5 (state |00⟩)")

In [ ]:
# Create VQE solver with depth=2 ansatz
vqe = VQE(H, ansatz_depth=2)
print(f"Number of qubits: {vqe.n_qubits}")
print(f"Number of parameters: {vqe.n_params}")

In [ ]:
# Run VQE optimization with SLSQP
print("Running VQE optimization (SLSQP)...")
ground_energy = vqe.run(optimizer='SLSQP', maxiter=200)
print(f"\n=== Results ===")
print(f"Ground-state energy: {ground_energy:.6f}")
print(f"Exact:               -1.500000")
print(f"Error:               {abs(ground_energy + 1.5):.2e}")
print(f"Optimal params: {np.round(vqe.optimal_params, 4)}")

In [ ]:
# Compare SLSQP vs SPSA convergence
vqe_spsa = VQE(H, ansatz_depth=2)
energy_spsa = vqe_spsa.run(optimizer='SPSA', maxiter=200)
print(f"SLSQP energy: {ground_energy:.6f}")
print(f"SPSA  energy: {energy_spsa:.6f}")
print(f"Exact:        -1.500000")

In [ ]:
# Plot convergence comparison
import matplotlib.pyplot as plt
plt.plot(vqe.energy_history, '-o', markersize=3, alpha=0.7, label='SLSQP')
plt.plot(vqe_spsa.energy_history, '-s', markersize=3, alpha=0.7, label='SPSA')
plt.axhline(y=-1.5, color='r', linestyle='--', label='Exact (-1.5)')
plt.xlabel('Optimizer iteration')
plt.ylabel('Energy')
plt.title('VQE Convergence: SLSQP vs SPSA')
plt.legend()
plt.grid(True)
plt.show()